# Predicción Libre de Animalitos (Sin restricciones)

Ensemble de modelos (RF + XGBoost + baseline de recencia), features calendarias, cíclicas y de recencia sin fuga; asignación Húngara para 12 horas sin repetición. Predice para mañana y exporta CSVs.


In [16]:
# Librerías
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier

import xgboost as xgb
from scipy.optimize import linear_sum_assignment

sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (12, 6)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

BASE_DIR = Path.cwd()
# Rutas candidatas (absoluta y relativa)
DATA_CANDIDATES = [
    Path("/home/samuelb/Documentos/python/Inteligencia_Artificial_Uni/Examen Final/Animalitos/resultados_guacharoactivo_completo2.txt"),
    BASE_DIR / "resultados_guacharoactivo_completo2.txt",
    Path("/home/samuelb/Documentos/python/Inteligencia_Artificial_Uni/Examen Final/Animalitos/resultados_guacharoactivo_completo.txt"),
    BASE_DIR / "resultados_guacharoactivo_completo.txt",
]
DATA_PATH = next((p for p in DATA_CANDIDATES if p.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("No se encontró el archivo de datos en rutas candidatas.")


# Carga y limpieza


In [17]:
df = pd.read_csv(
    DATA_PATH,
    names=["Fecha", "Hora", "Animal"],
    sep=",",
    dtype={"Fecha": str, "Hora": str, "Animal": str},
)

# Limpiar espacios
for col in ["Fecha", "Hora", "Animal"]:
    df[col] = df[col].astype(str).str.strip()

# Parseo de fechas (robusto)
df["Fecha"] = pd.to_datetime(df["Fecha"], format="%Y-%m-%d", errors="coerce")
if df["Fecha"].isna().mean() > 0.5:
    # Intento flexible si la mayoría falló
    df["Fecha"] = pd.to_datetime(df["Fecha"], errors="coerce")

# Parseo de hora (12h o 24h)
hora_str = df["Hora"].astype(str).str.strip()
hr12 = pd.to_datetime(hora_str, format="%I:%M %p", errors="coerce").dt.hour
hr24 = pd.to_datetime(hora_str, format="%H:%M", errors="coerce").dt.hour
hora_num = hr12.fillna(hr24)

# Filtrar válidos
df = df.loc[df["Fecha"].notna() & hora_num.notna()].copy()
df["Hora24"] = hora_num.loc[df.index].astype(int)

# Derivadas de calendario
df["DiaSemana"] = df["Fecha"].dt.dayofweek
df["Mes"] = df["Fecha"].dt.month
df["Año"] = df["Fecha"].dt.year
df["SemanaISO"] = df["Fecha"].dt.isocalendar().week.astype(int)
df["DiaMes"] = df["Fecha"].dt.day

# Etiquetas
le = LabelEncoder()
df["AnimalId"] = le.fit_transform(df["Animal"].values)
id_to_animal = {i: a for i, a in enumerate(le.classes_)}

print(df.head(3))
print("Clases:", len(le.classes_))


       Fecha      Hora    Animal  Hora24  DiaSemana  Mes   Año  SemanaISO  \
0 2022-12-26  09:00 AM  Pelicano       9          0   12  2022         52   
1 2022-12-26  10:00 AM       Oso      10          0   12  2022         52   
2 2022-12-26  11:00 AM    Perico      11          0   12  2022         52   

   DiaMes  AnimalId  
0      26        58  
1      26        50  
2      26        60  
Clases: 78


# Features: cíclicas + recencia (expanding counts shifted)

Construye señales de frecuencia previas sin fuga de información.


In [18]:
def add_cyclical(d: pd.DataFrame) -> pd.DataFrame:
    o = d.copy()
    o["sin_hora"] = np.sin(2 * np.pi * o["Hora24"] / 24)
    o["cos_hora"] = np.cos(2 * np.pi * o["Hora24"] / 24)
    o["sin_dow"] = np.sin(2 * np.pi * o["DiaSemana"] / 7)
    o["cos_dow"] = np.cos(2 * np.pi * o["DiaSemana"] / 7)
    o["sin_mes"] = np.sin(2 * np.pi * (o["Mes"] - 1) / 12)
    o["cos_mes"] = np.cos(2 * np.pi * (o["Mes"] - 1) / 12)
    return o

# Expanding counts previos (sin fuga): global, por hora, y por día-semana
# Ordenar crónicamente y usar shift(1) para no usar el presente
work = df.sort_values(["Fecha", "Hora24"]).copy()
work["one"] = 1

# Global por animal
work["cnt_prev_global"] = (
    work.groupby("AnimalId")["one"].cumsum().shift(1).fillna(0)
)
# Por hora (AnimalId, Hora24)
work["cnt_prev_hora"] = (
    work.groupby(["AnimalId", "Hora24"])["one"].cumsum().shift(1).fillna(0)
)
# Por día de semana (AnimalId, DiaSemana)
work["cnt_prev_dow"] = (
    work.groupby(["AnimalId", "DiaSemana"])["one"].cumsum().shift(1).fillna(0)
)

work = add_cyclical(work)

FEATURES = [
    "Hora24", "DiaSemana", "Mes", "Año", "SemanaISO", "DiaMes",
    "sin_hora", "cos_hora", "sin_dow", "cos_dow", "sin_mes", "cos_mes",
    "cnt_prev_global", "cnt_prev_hora", "cnt_prev_dow",
]

# Construcción de X, y con fallback si queda vacío
y = work["AnimalId"].values
X = work[FEATURES].astype(float).values
if X.shape[0] == 0:
    # Fallback minimal: solo features calendarias y cíclicas
    FEATURES = [
        "Hora24", "DiaSemana", "Mes", "Año", "SemanaISO", "DiaMes",
        "sin_hora", "cos_hora", "sin_dow", "cos_dow", "sin_mes", "cos_mes",
    ]
    if not {"sin_hora"}.issubset(work.columns):
        work = add_cyclical(work)
    y = work["AnimalId"].values
    X = work[FEATURES].astype(float).values

print("X shape:", X.shape)



X shape: (12036, 15)


# Entrenamiento: RF + XGBoost, y baseline de recencia por hora

- RF: rápido y robusto.
- XGB: booster con early stopping (API nativa).
- Baseline recencia: distribución por hora en los últimos 56 días (si no hay, usar todo el pasado).


In [19]:
# Split temporal simple para early stopping de XGB (robusto a datasets cortos)
work = work.sort_values(["Fecha", "Hora24"]).reset_index(drop=True)
unique_days = work["Fecha"].dt.date.unique()

# Heurística para tamaño de validación por días
n_days = len(unique_days)
val_days = 14 if n_days > 60 else min(7, max(1, n_days // 10))
if val_days >= n_days:
    val_days = max(1, n_days - 1)

def build_split(vdays: int):
    if vdays <= 0:
        mask = np.zeros(len(work), dtype=bool)
    else:
        vdates = set(unique_days[-vdays:])
        mask = work["Fecha"].dt.date.isin(vdates).values
    Xtr = work.loc[~mask, FEATURES].astype(float).values
    ytr = work.loc[~mask, "AnimalId"].values
    Xv = work.loc[mask, FEATURES].astype(float).values
    yv = work.loc[mask, "AnimalId"].values
    return mask, Xtr, ytr, Xv, yv

val_mask, X_tr, y_tr, X_val, y_val = build_split(val_days)

# Si no hay muestras de train, reduce validación; si sigue vacío, usa todo como train y duplica train como val
attempts = 3
while X_tr.shape[0] == 0 and attempts > 0:
    val_days = max(0, val_days // 2)
    val_mask, X_tr, y_tr, X_val, y_val = build_split(val_days)
    attempts -= 1

if X_tr.shape[0] == 0:
    # Fallback: usar todos los datos construidos X, y
    X_tr, y_tr = X, y
    X_val, y_val = X_tr, y_tr

if X_val.shape[0] == 0:
    X_val, y_val = X_tr, y_tr

# Si aun así no hay datos, abortar con mensaje claro
if X_tr.shape[0] == 0:
    raise ValueError("No hay muestras para entrenar tras construir features. Revisa la carga de datos.")

# RF
rf = RandomForestClassifier(
    n_estimators=600,
    max_depth=None,
    min_samples_leaf=2,
    max_features="sqrt",
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
print("Entrenando RF...")
rf.fit(X_tr, y_tr)

# XGB (API nativa)
print("Entrenando XGB...")
dtrain = xgb.DMatrix(X_tr, label=y_tr)
dval = xgb.DMatrix(X_val, label=y_val)
num_class = len(le.classes_)
params = {
    'objective': 'multi:softprob',
    'eval_metric': 'mlogloss',
    'num_class': int(num_class),
    'max_depth': 6,
    'eta': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 2,
    'lambda': 2.0,
    'tree_method': 'hist',
}
watchlist = [(dval, 'validation')]
xgb_booster = xgb.train(
    params,
    dtrain,
    num_boost_round=1500,
    evals=watchlist,
    early_stopping_rounds=50,
    verbose_eval=False,
)
print("Best iter XGB:", getattr(xgb_booster, 'best_iteration', 'N/A'))

# Baseline de recencia: prob(animal | hora, últimas 56 jornadas)
window_days = 56
last_date = work["Fecha"].max().date()
cut_date = pd.Timestamp(last_date - timedelta(days=window_days))
recent = work[work["Fecha"] >= cut_date]
if recent.empty:
    recent = work

# Matriz hora x clase con frecuencias normalizadas
probs_baseline = {}
for h in range(8, 20):
    sub = recent[recent["Hora24"] == h]
    counts = sub["AnimalId"].value_counts().reindex(range(num_class), fill_value=0).values
    p = counts / counts.sum() if counts.sum() > 0 else np.ones(num_class) / num_class
    probs_baseline[h] = p

print("Entrenamiento listo.")


Entrenando RF...
Entrenando XGB...
Best iter XGB: 3
Entrenamiento listo.


# Predicción de mañana con ensemble y asignación Húngara

- Ensemble: promedio ponderado de probabilidades (RF 0.4, XGB 0.4, Baseline 0.2).
- Salida: 12 animales únicos (08–19). Exporta CSV.


In [20]:
# Construir frame de mañana
ultima_fecha = df["Fecha"].max().date()
fecha_objetivo = ultima_fecha + timedelta(days=1)
rows = []
for h in range(8, 20):
    rows.append({
        "Fecha": pd.Timestamp(fecha_objetivo),
        "Hora24": h,
        "DiaSemana": pd.Timestamp(fecha_objetivo).dayofweek,
        "Mes": pd.Timestamp(fecha_objetivo).month,
        "Año": pd.Timestamp(fecha_objetivo).year,
        "SemanaISO": pd.Timestamp(fecha_objetivo).isocalendar().week,
        "DiaMes": pd.Timestamp(fecha_objetivo).day,
    })

fut = pd.DataFrame(rows)
# Agregar recencia para mañana a partir del histórico (última fila por hora)
hist = work.sort_values(["Hora24", "Fecha"]).copy()
last_per_hour = hist.groupby("Hora24").tail(1)[["Hora24", "cnt_prev_global", "cnt_prev_hora", "cnt_prev_dow"]]
fut = fut.merge(last_per_hour, on="Hora24", how="left")

fut = add_cyclical(fut)
X_manana = fut[FEATURES].astype(float).values

# Prob RF y XGB
dm = xgb.DMatrix(X_manana)
proba_rf = rf.predict_proba(X_manana)
proba_xgb = xgb_booster.predict(dm)

# Prob baseline por hora (tomar en orden de horas 8..19)
proba_base = np.vstack([probs_baseline[h] for h in range(8, 20)])

# Ensemble
w_rf, w_xgb, w_base = 0.4, 0.4, 0.2
proba_ens = w_rf * proba_rf + w_xgb * proba_xgb + w_base * proba_base

# Asignación Húngara
costs = -np.log(np.clip(proba_ens, 1e-9, 1.0))
rows_idx, cols_idx = linear_sum_assignment(costs)

pred = []
for i, (r, c) in enumerate(zip(rows_idx, cols_idx)):
    pred.append({
        "Fecha": fecha_objetivo,
        "Hora": f"{(8+r):02d}:00",
        "Animal": id_to_animal[c],
        "Probabilidad": float(proba_ens[r, c])
    })

pred_df = pd.DataFrame(pred).sort_values("Hora").reset_index(drop=True)

# Exportar
out = BASE_DIR / f"predicciones_libre_{fecha_objetivo.strftime('%Y%m%d')}_ENS.csv"
pred_df.to_csv(out, index=False)
print("Exportado:", out)
pred_df.head(12)


Exportado: /home/samuelb/Documentos/python/Inteligencia_Artificial_Uni/Examen Final/Animalitos/predicciones_libre_20251030_ENS.csv


,Fecha,Hora,Animal,Probabilidad
0,2025-10-30,08:00,Jaguar,0.028531
1,2025-10-30,09:00,Camello,0.028822
2,2025-10-30,10:00,Cebra,0.049639
3,2025-10-30,11:00,Ballena,0.036636
4,2025-10-30,12:00,Tiburon,0.037110
5,2025-10-30,13:00,Chivo,0.040987
6,2025-10-30,14:00,Canario,0.024011
7,2025-10-30,15:00,Tucan,0.025342
8,2025-10-30,16:00,Gorila,0.039564
9,2025-10-30,17:00,Grillo,0.033310
